# Itertools et générateurs

Ce notebook couvre deux notions complémentaires :
- Les **générateurs** : un mécanisme Python pour produire des séquences de valeurs **à la demande**, sans tout stocker en mémoire
- Le module **itertools** : une boîte à outils de fonctions pour manipuler des séquences (combiner, filtrer, accumuler...)

Ces outils sont utiles en data science quand on travaille avec de grands volumes de données ou qu'on veut écrire du code concis et efficace.

In [1]:
import itertools

## Les générateurs

Un générateur est une fonction spéciale qui produit une séquence de valeurs **une par une**, au lieu de les calculer toutes d'un coup et de les stocker en mémoire.

La différence avec une fonction classique :
- Une fonction normale utilise `return` : elle calcule un résultat, le renvoie, et **oublie tout** de son état interne
- Un générateur utilise `yield` : il produit une valeur, **suspend son exécution**, et reprend exactement là où il s'était arrêté quand on lui demande la valeur suivante

C'est comme un livre dont on tourne les pages une par une, au lieu de photocopier tout le livre d'un coup.

### Exemple simple

In [2]:
def simple_generator_function():
    yield 10
    yield 20
    yield 30
    

print("---- Generator 1 ----")
generator1 =   simple_generator_function()  
for i,value in enumerate(generator1):
    print("%d:\t" % i,value)
    
    
print("---- Generator 1 (bis) ----")
# Pourquoi aucun sortie ici ?
for i,value in enumerate(generator1):
    print("%d:\t" % i,value)
    
print("---- Generator 2 ----")
# Pourquoi une sortie ici ?
generator2 =   simple_generator_function()  
for i,value in enumerate(generator2):
    print("%d:\t" % i,value)
    
     

---- Generator 1 ----
0:	 10
1:	 20
2:	 30
---- Generator 1 (bis) ----
---- Generator 2 ----
0:	 10
1:	 20
2:	 30


**Observation clé** : le "Generator 1 (bis)" n'affiche rien. C'est parce qu'un générateur **s'épuise** après un parcours complet. Une fois toutes les valeurs produites, il est vide — comme un distributeur de tickets dont on a tiré le dernier.

Pour réobtenir les valeurs, il faut **créer un nouveau générateur** (c'est ce que fait "Generator 2" en rappelant `simple_generator_function()`). On ne peut pas "rembobiner" un générateur.

### Exemple réaliste : un générateur infini

L'intérêt majeur des générateurs, c'est qu'ils peuvent produire des séquences **infinies** sans exploser la mémoire. L'exemple ci-dessous génère tous les nombres premiers — une séquence qui ne s'arrête jamais. Le `yield` suspend la fonction à chaque nombre premier trouvé, et la reprend quand on demande le suivant :

In [3]:
import math
def is_prime(n):
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

def all_primes():
    k = 2
    while True:
        if is_prime(k):
            yield k # suspend la fonction, retourne un générateur (iterable) -> casse la boucle (while True) 
        k += 1

In [4]:
# i représente une variable d'itération incrémentée à chaque nouvelle valeur retournée par le generateur 'all_primes'.
# enumerate(all_primes) est appelé à chaque itération et le résultat qu'il retourne est affecté à la variable n. 
# Cette méthode reprend exactement là où elle avait arrétée (yield k). On peut dire que n = k ...
# Attention, la boucle ci-dessous est infinie ! il faut impérativement mettre une clause de sortie
for i, n in enumerate(all_primes()):
    if i > 10:
        break
    print(n)

2
3
5
7
11
13
17
19
23
29
31


## Le module itertools

On vient de voir comment créer nos propres générateurs avec `yield`. Le module `itertools` de la bibliothèque standard fournit des **générateurs prêts à l'emploi** pour des opérations courantes : compter, découper, combiner, filtrer, accumuler...

Comme les générateurs, ces fonctions sont **paresseuses** (lazy) : elles ne calculent les valeurs que quand on les demande. Pas de gaspillage de mémoire.

### count()

`count(start, step)` crée un compteur **infini** qui commence à `start` et avance de `step` en `step` (par défaut step=1).

Attention : comme la séquence est infinie, il **faut** prévoir une condition de sortie (`break`) dans votre boucle, sinon elle tourne indéfiniment.

In [5]:
from itertools import count
print("\n-- Sans step")        
for i in count(10):
    if i > 15:
        break
    else:
        print(i)
        
print("\n-- Avec step")        
for i in count(10,2):
    if i > 15:
        break
    else:
        print(i)


-- Sans step
10
11
12
13
14
15

-- Avec step
10
12
14


### islice()

`islice(iterable, stop)` permet de **limiter** le nombre d'éléments extraits d'un itérable. C'est l'équivalent du slicing `[:n]` sur les listes, mais qui fonctionne aussi sur les générateurs et les itérateurs infinis.

Combiné avec `count()`, il évite d'avoir à écrire un `break` manuel :

In [6]:
from itertools import islice
for i in islice(count(10), 5):
     print(i)
      

10
11
12
13
14


## accumulate()

`accumulate(iterable, operator)` applique un opérateur **de façon cumulative** sur les éléments d'une séquence, en gardant tous les résultats intermédiaires.

Le principe, pas à pas :

```
accumulate([a, b, c, d], op)  →  a, op(a,b), op(op(a,b),c), op(op(op(a,b),c),d)
```

Pensez-y comme un "total cumulé" : chaque étape combine le résultat précédent avec l'élément suivant.

In [7]:
from itertools import accumulate
from operator import add, mul, concat

data = range(5) # contient une liste allant de 0 à 5

In [8]:
# X = data[0]
# X = add(X, data[1])
# X = add(X, data[2])
# X = add(X, data[3])
# ...

for x in accumulate(data,add):
    print(x)

0
1
3
6
10


Avec la multiplication, on utilise `data[1:]` (en excluant le 0 du début) — sinon le premier élément serait 0 et tous les produits resteraient à 0. L'accumulation fait : 1, 1×2=2, 2×3=6, 6×4=24.

In [9]:
# X = data[1]
# X = mul(X, data[2])
# X = mul(X, data[3])
# ...
for x in accumulate(data[1:], mul):
    print(x)

1
2
6
24


In [10]:
# Pour faciliter la compréhension :
for x in ([y] for y in data):
    print(x)
print("----")

# Accumulate :
# X = data[0]   //  [0]
# X = concat(X, data[1]) // concat([0],[1])
# X = concat(X, data[2]) // concat([0,1],[2])
# ...
for x in accumulate(([y] for y in data), concat):
    print(x)

[0]
[1]
[2]
[3]
[4]
----
[0]
[0, 1]
[0, 1, 2]
[0, 1, 2, 3]
[0, 1, 2, 3, 4]


Les trois exemples ci-dessus montrent la souplesse de `accumulate` : on peut accumuler des sommes, des produits, ou même construire des listes progressivement — tout dépend de l'opérateur qu'on fournit.

## chain()

`chain(iterable1, iterable2, ...)` concatène plusieurs itérables en un seul, bout à bout. C'est l'équivalent du `+` sur les listes, mais sans créer de copie en mémoire.

Attention au **faux ami** : itérer directement sur un tuple de listes `(l1, l2, l3)` parcourt les listes elles-mêmes (3 itérations), pas leurs éléments. `chain` parcourt les éléments de chaque liste successivement :

In [11]:
from itertools import chain

l1 = ['foo', 'bar']
l2 = list(range(5))
l3 = ['ls', '/some/dir']

# Faux ami :
for val in (l1,l2,l3):
    print(val)
    
print("---------")

for val in chain(l1,l2,l3):
    print(val)


['foo', 'bar']
[0, 1, 2, 3, 4]
['ls', '/some/dir']
---------
foo
bar
0
1
2
3
4
ls
/some/dir


## combinations() et combinations_with_replacement()

`combinations(iterable, r)` génère tous les **sous-ensembles uniques** de taille `r` à partir d'un itérable, **sans** répétition d'éléments et sans tenir compte de l'ordre.

Par exemple, avec `['a', 'b', 'c', 'd']` et `r=2` : on obtient `(a,b), (a,c), (a,d), (b,c), (b,d), (c,d)` — toutes les paires possibles sans doublons :

In [12]:
from itertools import combinations

data = ['a','b','c','d']
result1 = combinations(data, 2)
for each in result1:
    print(each)
    
result2 = itertools.combinations(data, 3)
for each in result2:
    print(each)

('a', 'b')
('a', 'c')
('a', 'd')
('b', 'c')
('b', 'd')
('c', 'd')
('a', 'b', 'c')
('a', 'b', 'd')
('a', 'c', 'd')
('b', 'c', 'd')


`combinations_with_replacement(iterable, r)` fait la même chose, mais **autorise la répétition** d'un même élément. Avec `r=3`, on peut avoir `(a, a, a)`, `(a, a, b)`, etc. :

In [13]:
from itertools import combinations_with_replacement

data = ['a','b','c','d']
result = combinations_with_replacement(data, 3)
for each in result:
    print(each)


('a', 'a', 'a')
('a', 'a', 'b')
('a', 'a', 'c')
('a', 'a', 'd')
('a', 'b', 'b')
('a', 'b', 'c')
('a', 'b', 'd')
('a', 'c', 'c')
('a', 'c', 'd')
('a', 'd', 'd')
('b', 'b', 'b')
('b', 'b', 'c')
('b', 'b', 'd')
('b', 'c', 'c')
('b', 'c', 'd')
('b', 'd', 'd')
('c', 'c', 'c')
('c', 'c', 'd')
('c', 'd', 'd')
('d', 'd', 'd')


## compress()

`compress(data, selectors)` filtre les éléments de `data` en ne gardant que ceux dont le sélecteur correspondant est `True`. C'est l'équivalent du boolean indexing de NumPy, mais pour des listes classiques :

In [14]:
data = ['a','b','c','d']
selections = [True, False, True, False]
result = itertools.compress(data, selections)
for each in result:
    print(each)

a
c


## dropwhile()

`dropwhile(predicate, iterable)` **ignore** les éléments du début de la séquence tant que la condition `predicate` est vraie, puis retourne **tout le reste** (même si la condition redevient vraie plus tard).

Exemple : avec `lambda x: x < 5` sur `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1]`, les éléments 1-4 sont ignorés (< 5), puis tout à partir de 5 est retourné — y compris le `1` final :

In [15]:
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1]
result = itertools.dropwhile(lambda x: x<5, data)
for each in result:
    print(each)

5
6
7
8
9
10
1


> **En résumé :** Les générateurs (`yield`) produisent des valeurs à la demande sans consommer de mémoire. Le module `itertools` fournit des générateurs prêts à l'emploi pour les opérations courantes : compter (`count`), découper (`islice`), accumuler (`accumulate`), concaténer (`chain`), combiner (`combinations`), filtrer (`compress`, `dropwhile`). Ces outils rendent votre code plus concis et plus efficace sur de grands volumes de données.